# Prompt 17: Joins
## Databricks Certified Associate Developer for Apache Spark
### Topic 3 — DataFrame and Column Operations (20%)

---

## Join Syntax — Three Forms

```python
# Form 1: string column name (used when both DataFrames share the column name)
df1.join(df2, on='key', how='inner')

# Form 2: list of strings (multi-column join on shared names)
df1.join(df2, on=['key1', 'key2'], how='inner')

# Form 3: Column expression (required when column names differ, or for non-equi joins)
df1.join(df2, on=df1['id'] == df2['customer_id'], how='inner')
df1.join(df2, on=(df1['dept'] == df2['dept_id']) & (df1['year'] == df2['year']), how='inner')
```

## All Join Types

| `how` value | Aliases | Behaviour |
|-------------|---------|----------|
| `'inner'` | — | Only rows with matching keys on BOTH sides |
| `'left'` | `'left_outer'`, `'leftouter'` | All left rows; matching right rows or NULLs |
| `'right'` | `'right_outer'`, `'rightouter'` | All right rows; matching left rows or NULLs |
| `'outer'` | `'full'`, `'full_outer'`, `'fullouter'` | All rows from both sides; NULLs for non-matches |
| `'left_semi'` | `'leftsemi'` | Only left rows that HAVE a match in right (no right columns) |
| `'left_anti'` | `'leftanti'` | Only left rows that DO NOT have a match in right |
| `'cross'` | `'cartesian'` | Every left row × every right row (no condition needed) |

## Duplicate Column Problem

When joining on a **Column expression** (`df1['id'] == df2['id']`), **both** columns appear in the result:
```python
result = df1.join(df2, df1['id'] == df2['id'])  # two 'id' columns in result!
result.select('id')  # AnalysisException: ambiguous reference
```

**Solutions:**
```python
# Solution 1: use string/list 'on' (preferred — keeps only one key column)
df1.join(df2, on='id')                    # single 'id' column in result

# Solution 2: drop duplicate after join
df1.join(df2, df1['id'] == df2['id']).drop(df2['id'])

# Solution 3: rename before join
df2_renamed = df2.withColumnRenamed('id', 'right_id')
df1.join(df2_renamed, df1['id'] == df2_renamed['right_id'])

# Solution 4: alias the DataFrames
df1.alias('l').join(df2.alias('r'), col('l.id') == col('r.id')) \
   .select('l.id', 'l.name', 'r.salary')
```

In [ ]:
# Cell 1: Setup
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.functions import col, broadcast

spark = SparkSession.builder \
    .appName('Joins') \
    .master('local[*]') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

# Employees
employees = spark.createDataFrame([
    (1, 'Alice',   10, 95000),
    (2, 'Bob',     20, 72000),
    (3, 'Carol',   10, 88000),
    (4, 'Dave',    30, 65000),
    (5, 'Eve',     99, 91000),   # dept 99 does not exist
    (6, 'Frank',   20, 78000),
], ['emp_id', 'name', 'dept_id', 'salary'])

# Departments
departments = spark.createDataFrame([
    (10, 'Engineering', 'US'),
    (20, 'Marketing',   'UK'),
    (30, 'HR',          'US'),
    (40, 'Finance',     'CA'),   # no employee in Finance
], ['dept_id', 'dept_name', 'region'])

# Projects
projects = spark.createDataFrame([
    (1, 'Alpha',  1),
    (2, 'Beta',   1),
    (3, 'Gamma',  2),
    (4, 'Delta',  7),  # emp 7 does not exist
], ['proj_id', 'proj_name', 'lead_emp_id'])

print('=== Employees ===')
employees.show()
print('=== Departments ===')
departments.show()
print('=== Projects ===')
projects.show()

In [ ]:
# Cell 2: inner, left, right, full outer joins

print('=== INNER JOIN — only matched rows ===')
# Eve (dept 99) and Finance (dept 40) are dropped
inner = employees.join(departments, on='dept_id', how='inner')
inner.show()
print(f'inner row count: {inner.count()}')  # 5 (Eve excluded, Finance excluded)

print('=== LEFT JOIN — all employees, NULLs for missing dept ===')
left = employees.join(departments, on='dept_id', how='left')
left.show()
print(f'left row count: {left.count()}')  # 6 (Eve gets NULLs for dept columns)

print('=== RIGHT JOIN — all departments, NULLs for missing employees ===')
right = employees.join(departments, on='dept_id', how='right')
right.show()
print(f'right row count: {right.count()}')  # 6 (Finance gets NULLs for emp columns)

print('=== FULL OUTER JOIN — all rows from both sides ===')
outer = employees.join(departments, on='dept_id', how='outer')
outer.show()
print(f'outer row count: {outer.count()}')  # 7 (Eve + Finance both appear)

# Also valid: how='full', how='full_outer', how='fullouter'
assert employees.join(departments, on='dept_id', how='full').count() == \
       employees.join(departments, on='dept_id', how='full_outer').count()
print('full / full_outer / outer aliases are equivalent')

In [ ]:
# Cell 3: left_semi and left_anti joins

print('=== LEFT SEMI JOIN — employees who have a department (filtering join) ===')
# Returns left rows that HAVE a match — NO columns from the right
semi = employees.join(departments, on='dept_id', how='left_semi')
semi.show()
print(f'semi columns: {semi.columns}')  # only employee columns — NO dept columns
print(f'semi row count: {semi.count()}')  # 5 (Eve excluded)

print('=== LEFT ANTI JOIN — employees with NO matching department ===')
# Returns left rows that have NO match in right — NO columns from the right
anti = employees.join(departments, on='dept_id', how='left_anti')
anti.show()
print(f'anti columns: {anti.columns}')  # only employee columns
print(f'anti row count: {anti.count()}')  # 1 (only Eve, dept 99 not found)

print('=== Semi vs Inner — key difference: semi produces NO right-side columns ===')
inner_cols = employees.join(departments, on='dept_id', how='inner').columns
semi_cols  = employees.join(departments, on='dept_id', how='left_semi').columns
print(f'inner columns: {inner_cols}')
print(f'semi columns:  {semi_cols}')

print('=== Real use case: semi as EXISTS / anti as NOT EXISTS ===')
# Find departments that have at least one employee (EXISTS equivalent)
depts_with_employees = departments.join(employees, on='dept_id', how='left_semi')
depts_with_employees.show()

# Find departments with NO employees (NOT EXISTS equivalent)
depts_no_employees = departments.join(employees, on='dept_id', how='left_anti')
depts_no_employees.show()

In [ ]:
# Cell 4: Cross join and non-equi joins

print('=== CROSS JOIN — cartesian product ===')
# Every employee row paired with every department row
cross = employees.crossJoin(departments)
print(f'employees: {employees.count()}, departments: {departments.count()}')
print(f'cross join rows: {cross.count()}')  # 6 * 4 = 24
cross.show(5)

# Alternatively: join with how='cross'
cross2 = employees.join(departments, how='cross')
print(f'cross2 rows: {cross2.count()}')  # same 24 rows

print('=== Non-equi join — salary within department band ===')
salary_bands = spark.createDataFrame([
    ('junior',  0,      80000),
    ('senior',  80001,  100000),
    ('lead',    100001, 999999),
], ['level', 'min_sal', 'max_sal'])

# Find level for each employee
classified = employees.join(
    salary_bands,
    (col('salary') >= col('min_sal')) & (col('salary') <= col('max_sal')),
    how='left'
)
classified.select('name', 'salary', 'level').show()

print('=== Multi-column join ===')
# Join on multiple columns using a list of strings
data1 = spark.createDataFrame([(1,'US',100),(2,'UK',200)], ['id','region','val1'])
data2 = spark.createDataFrame([(1,'US','A'),(2,'CA','B'),(1,'UK','C')], ['id','region','val2'])
data1.join(data2, on=['id', 'region'], how='inner').show()

## Broadcast Join

A **broadcast join** avoids a shuffle by sending the entire small table to every executor.
It is the most important join optimisation for fact-table + small-dimension-table patterns.

### Automatic broadcast (AQE and static threshold)
```python
# Spark automatically broadcasts tables below this byte threshold:
spark.conf.get('spark.sql.autoBroadcastJoinThreshold')   # default: 10MB (10485760)
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', 50 * 1024 * 1024)  # 50 MB
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', -1)  # disable auto-broadcast
```

### Explicit broadcast hint
```python
from pyspark.sql.functions import broadcast

# Hint Spark to broadcast the right (small) table
df_large.join(broadcast(df_small), on='key', how='inner')

# SQL equivalent
spark.sql('SELECT /*+ BROADCAST(dim) */ * FROM fact f JOIN dim d ON f.id = d.id')
```

### When to broadcast
- Right (or left) side is a small lookup/dimension table (< ~10–100 MB)
- Avoids an expensive shuffle of the large table
- Especially important when auto-broadcast is disabled or threshold is too low

### Verify broadcast in query plan
```python
df_large.join(broadcast(df_small), on='key').explain()  # look for BroadcastHashJoin
```

## Handling Duplicate Column Names After Join

```python
# Problem: Column expression join creates two 'dept_id' columns
result = employees.join(departments, employees['dept_id'] == departments['dept_id'])
result.columns  # ['emp_id', 'name', 'dept_id', 'salary', 'dept_id', 'dept_name', 'region']

# Fix 1: use string 'on' to merge the key column (best for equi-joins)
employees.join(departments, on='dept_id')   # single 'dept_id' in result

# Fix 2: drop the duplicate BEFORE or AFTER join using DataFrame reference
employees.join(departments, employees['dept_id'] == departments['dept_id']) \
         .drop(departments['dept_id'])   # must use DataFrame ref, not string

# Fix 3: alias DataFrames and qualify references
employees.alias('e').join(departments.alias('d'), col('e.dept_id') == col('d.dept_id')) \
         .select('e.emp_id', 'e.name', 'd.dept_name', 'd.region')

# Fix 4: rename before join
departments.withColumnRenamed('dept_id', 'd_dept_id') \
           .join(employees, col('emp_id') ...)
```

In [ ]:
# Cell 5: Broadcast join and duplicate column handling

print('=== Broadcast join hint ===')
# departments is small — broadcast it to all executors
result = employees.join(broadcast(departments), on='dept_id', how='inner')
result.show()

print('=== Verify broadcast in query plan (look for BroadcastHashJoin) ===')
employees.join(broadcast(departments), on='dept_id').explain()

print('=== Compare: SortMergeJoin vs BroadcastHashJoin ===')
# Disable auto-broadcast to force SortMergeJoin
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')
employees.join(departments, on='dept_id').explain()  # SortMergeJoin

# Re-enable
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '10485760')

print('=== Duplicate column problem with Column expression join ===')
bad_join = employees.join(departments,
                          employees['dept_id'] == departments['dept_id'],
                          how='inner')
print(f'columns with expression join: {bad_join.columns}')  # two dept_id columns
# bad_join.select('dept_id').show()  # AnalysisException: ambiguous

print('=== Fix 1: string on — single key column ===')
fix1 = employees.join(departments, on='dept_id', how='inner')
print(f'columns with string on: {fix1.columns}')  # one dept_id
fix1.show()

print('=== Fix 2: drop duplicate via DataFrame reference ===')
fix2 = employees.join(departments,
                       employees['dept_id'] == departments['dept_id'],
                       how='inner') \
                .drop(departments['dept_id'])
print(f'columns after drop(departments[dept_id]): {fix2.columns}')

print('=== Fix 3: alias DataFrames ===')
fix3 = employees.alias('e') \
                .join(departments.alias('d'),
                      col('e.dept_id') == col('d.dept_id'),
                      how='inner') \
                .select('e.emp_id', 'e.name', 'e.salary',
                        'd.dept_name', 'd.region')
fix3.show()

In [ ]:
# Cell 6: Chained joins, SQL join syntax, and join with aggregation

print('=== Chained joins — employees + departments + projects ===')
# Join employees to departments, then join the result to projects
chained = employees \
    .join(departments, on='dept_id', how='left') \
    .join(
        projects.withColumnRenamed('lead_emp_id', 'emp_id'),
        on='emp_id',
        how='left'
    ) \
    .select('name', 'dept_name', 'region', 'proj_name', 'salary')
chained.show()

print('=== SQL join syntax ===')
employees.createOrReplaceTempView('emp')
departments.createOrReplaceTempView('dept')
projects.createOrReplaceTempView('proj')

spark.sql("""
    SELECT e.name, d.dept_name, d.region, e.salary
    FROM   emp e
    LEFT   JOIN dept d ON e.dept_id = d.dept_id
    WHERE  e.salary > 75000
    ORDER  BY e.salary DESC
""").show()

print('=== Join then aggregate — employees per department ===')
employees.join(departments, on='dept_id', how='inner') \
         .groupBy('dept_name', 'region') \
         .agg(
             F.count('emp_id').alias('headcount'),
             F.avg('salary').alias('avg_salary'),
             F.max('salary').alias('max_salary'),
         ) \
         .orderBy('dept_name') \
         .show()

print('=== Join with filter on result ===')
employees.join(departments, on='dept_id', how='inner') \
         .filter(col('region') == 'US') \
         .select('name', 'dept_name', 'salary') \
         .show()

spark.stop()
print('Done.')

## Quick Reference Summary

### Join syntax
```python
df1.join(df2, on='key', how='inner')               # string — merges key column
df1.join(df2, on=['k1', 'k2'], how='inner')        # list — multi-column
df1.join(df2, df1['a'] == df2['b'], how='inner')   # expression — keeps both columns
```

### Join types
```python
how='inner'        # matched rows only
how='left'         # all left + nulls for unmatched right
how='right'        # all right + nulls for unmatched left
how='outer'        # all rows from both sides
how='left_semi'    # left rows that HAVE match (no right columns)
how='left_anti'    # left rows that DO NOT have match
how='cross'        # cartesian product — every × every
```

### Broadcast join
```python
from pyspark.sql.functions import broadcast
df_large.join(broadcast(df_small), on='key')       # explicit hint
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', 50*1024*1024)  # raise threshold
```

### Duplicate column resolution
```python
df1.join(df2, on='key')                            # string on — no duplicate
.drop(df2['col'])                                  # drop by DataFrame ref
df1.alias('l').join(df2.alias('r'), ...).select('l.col', 'r.col')  # alias qualify
```

## Real-World Scenarios

<details>
<summary>Scenario 1: Fact-dimension star schema join with broadcast</summary>

**Situation:**
A sales fact table (500M rows) must be enriched with product and store dimension tables (< 1 MB each).

**Code:**
```python
enriched = sales_fact \
    .join(broadcast(product_dim), on='product_id', how='left') \
    .join(broadcast(store_dim),   on='store_id',   how='left') \
    .join(broadcast(date_dim),    on='date_id',    how='left') \
    .select(
        'sale_id', 'amount',
        'product_name', 'category',
        'store_name', 'region',
        'date', 'year', 'quarter'
    )

enriched.write.mode('overwrite').parquet('/warehouse/enriched_sales')
```

**Exam Sub-topic:** `broadcast()` hint for small dimension tables; chained left joins; BroadcastHashJoin vs SortMergeJoin
</details>

<details>
<summary>Scenario 2: semi/anti join — incremental ETL (process only new records)</summary>

**Situation:**
A daily batch loads new customer records. Use anti-join to identify records not yet in the warehouse.

**Code:**
```python
# existing warehouse records (just need the key)
existing_keys = warehouse.select('customer_id')

# New records not yet in warehouse (anti-join = NOT IN / NOT EXISTS)
new_records = incoming.join(existing_keys, on='customer_id', how='left_anti')

# Records that already exist and need updating (semi-join = IN / EXISTS)
updates = incoming.join(existing_keys, on='customer_id', how='left_semi')

print(f'New records to insert: {new_records.count()}')
print(f'Existing records to update: {updates.count()}')

new_records.write.mode('append').delta('/warehouse/customers')
```

**Exam Sub-topic:** `left_anti` = NOT IN; `left_semi` = IN; no right-side columns in result
</details>

<details>
<summary>Scenario 3: Full outer join — reconcile two system extracts</summary>

**Situation:**
Two systems export their customer records. A full outer join identifies records present in
system A only, system B only, or both — for data reconciliation.

**Code:**
```python
reconciled = system_a.alias('a') \
    .join(system_b.alias('b'),
          col('a.customer_id') == col('b.customer_id'),
          how='outer') \
    .withColumn('source',
        F.when(col('a.customer_id').isNull(), lit('B only'))
         .when(col('b.customer_id').isNull(), lit('A only'))
         .otherwise(lit('Both'))
    )

reconciled.groupBy('source').count().show()
reconciled.filter(col('source') != 'Both').show()  # discrepancies
```

**Exam Sub-topic:** full outer join; `when().isNull()` pattern to classify unmatched rows
</details>

<details>
<summary>Scenario 4: Non-equi join — assign pricing tier based on quantity range</summary>

**Situation:**
Order lines must be assigned a discount tier based on order quantity falling within a range.

**Code:**
```python
# Tiers table: tier_name, min_qty, max_qty, discount_pct
order_lines_with_tier = order_lines.join(
    discount_tiers,
    (col('quantity') >= col('min_qty')) & (col('quantity') <= col('max_qty')),
    how='left'
).withColumn(
    'discounted_price',
    col('unit_price') * (1 - F.coalesce(col('discount_pct'), lit(0.0)))
)

order_lines_with_tier.select('order_id', 'quantity', 'tier_name',
                              'discount_pct', 'discounted_price').show()
```

**Exam Sub-topic:** non-equi join with Column expression; range condition `>=` and `<=`
</details>

<details>
<summary>Scenario 5: Self-join — employee manager hierarchy</summary>

**Situation:**
An employee table has a `manager_id` foreign key referencing `emp_id` in the same table.
Use a self-join with aliases to retrieve each employee's manager name.

**Code:**
```python
emp = spark.table('employees')  # columns: emp_id, name, manager_id, salary

employee_with_manager = emp.alias('e') \
    .join(
        emp.alias('m'),
        col('e.manager_id') == col('m.emp_id'),
        how='left'
    ) \
    .select(
        col('e.emp_id'),
        col('e.name').alias('employee'),
        col('m.name').alias('manager'),
        col('e.salary'),
    )

employee_with_manager.show()
```

**Exam Sub-topic:** self-join using `.alias()` on the same DataFrame; qualify columns with alias prefix
</details>

## Exam Cheat Sheet

| Concept | Key Exam Fact |
|---------|---------------|
| `how='inner'` default | Returns only rows with keys present on BOTH sides |
| `how='left'` aliases | `'left_outer'`, `'leftouter'` — all equivalent |
| `how='outer'` aliases | `'full'`, `'full_outer'`, `'fullouter'` — all equivalent |
| `left_semi` result | Only **left** columns — right columns are **not** in output |
| `left_anti` result | Only **left** columns — rows with NO match in right |
| `left_semi` SQL equivalent | `WHERE key IN (SELECT key FROM right)` |
| `left_anti` SQL equivalent | `WHERE key NOT IN (SELECT key FROM right)` |
| `cross` join | No condition — produces left_rows × right_rows rows |
| String `on='key'` | Produces **one** key column in result |
| Column expr join | Produces **two** key columns — can cause `AnalysisException` |
| Fix for duplicate cols | Use string `on`, or `.drop(df2['col'])`, or `.alias()` |
| `broadcast(df)` | Sends small table to all executors — no shuffle |
| `autoBroadcastJoinThreshold` | Default 10 MB — tables smaller are auto-broadcast |
| Disable auto-broadcast | Set threshold to `-1` |
| Broadcast plan name | `BroadcastHashJoin` in `explain()` output |
| SortMergeJoin | Used for large-large joins — requires shuffle + sort |
| NULL join keys | **Never matched** in standard joins |
| Null-safe join | `df1['key'].eqNullSafe(df2['key'])` |
| Non-equi join | Use Column expression: `col('a') >= col('b')` |
| Self-join | Requires `.alias()` on both sides to disambiguate columns |

---

## Practice Q&A

<details>
<summary>Q1: What is the difference between left_semi and inner join?</summary>

**A:** Both return only rows from the left side that have a match in the right side. The critical difference is:
- **`inner`** — includes columns from **both** sides in the result.
- **`left_semi`** — includes columns from the **left side only**. The right table is used only for filtering, not for output.

`left_semi` is equivalent to `WHERE key IN (SELECT key FROM right)` in SQL and is useful when you want to filter a table based on membership in another without adding extra columns.
</details>

<details>
<summary>Q2: When does joining with a Column expression create duplicate columns, and how do you fix it?</summary>

**A:** When you join using a Column expression (`df1['col'] == df2['col']`), Spark keeps **both** columns in the output since they may technically hold different values in outer joins. Selecting by column name then causes `AnalysisException: Ambiguous column name`.

Three fixes:
1. **Preferred:** Use string `on='col'` — Spark merges the key column, keeping only one copy.
2. Drop the duplicate: `.drop(df2['col'])` — must use the DataFrame reference, not a string.
3. Alias DataFrames: `.alias('l')` / `.alias('r')` and qualify with `col('l.col')` / `col('r.col')`.
</details>

<details>
<summary>Q3: What is a broadcast join and when should you use it?</summary>

**A:** A broadcast join avoids a shuffle by sending the entire small table to every executor node, so each executor can perform the join locally against its partition of the large table.

Use it when:
- One side is a small lookup/dimension table (typically < 10–100 MB).
- You want to avoid the expensive shuffle+sort of `SortMergeJoin`.

Apply it with `df_large.join(broadcast(df_small), on='key')`. Spark also auto-broadcasts tables below `spark.sql.autoBroadcastJoinThreshold` (default 10 MB).

Verify in the plan: `explain()` will show `BroadcastHashJoin` instead of `SortMergeJoin`.
</details>

<details>
<summary>Q4: What does left_anti join return?</summary>

**A:** `left_anti` returns all rows from the **left** DataFrame that have **no matching key** in the right DataFrame. Only the left side's columns appear in the result.

SQL equivalent: `WHERE key NOT IN (SELECT key FROM right)` or `WHERE NOT EXISTS (...)`.

Common use: incremental ETL — find new records in a source that don't yet exist in the target warehouse.
</details>

<details>
<summary>Q5: How do full outer, left, and right joins differ for unmatched rows?</summary>

**A:**
- **`left`** — all rows from the **left** side are kept. For left rows with no match, right-side columns are `NULL`. Right rows with no match are **dropped**.
- **`right`** — all rows from the **right** side are kept. For right rows with no match, left-side columns are `NULL`. Left rows with no match are **dropped**.
- **`outer` / `full`** — all rows from **both** sides are kept. Non-matching rows from either side get `NULL` for the other side's columns.

Memory aid: the named side (`left` / `right`) always survives; the unnamed side can be dropped.
</details>